In [ ]:
"""
Week 1 - Day 4
Q-Table Deep Analysis
======================
Understanding what Q-Learning learned
about optimal pricing strategy.

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import sys

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)
from utils.q_table_analyzer import (
    analyze_q_table,
    plot_policy_heatmaps,
    analyze_learned_behavior
)
from utils.policy_extractor import (
    extract_policy_table,
    save_policy_summary
)

plt.style.use('seaborn-v0_8')
print("✅ Analysis modules loaded!")

In [ ]:
env   = DynamicPricingEnv()
agent = QLearningAgent(env, QL_CONFIG)

print("Training Q-Learning (5000 episodes)...")
rewards = agent.train(
    n_episodes=5000,
    verbose=True
)
print(f"\n✅ Trained! Final mean: "
      f"${np.mean(rewards[-100:]):.0f}")

In [ ]:
results = analyze_q_table(agent)

In [ ]:
plot_policy_heatmaps(
    agent,
    save_path='../results/policy_analysis.png'
)

In [ ]:
analyze_learned_behavior(agent)

In [ ]:
policy_df = extract_policy_table(agent)

print("=== POLICY TABLE (Sample) ===\n")
print(policy_df.groupby(
    'Category'
)['Optimal Price'].mean().round(0))

print("\n  Full policy sample:")
print(policy_df.head(20).to_string(index=False))

In [ ]:
summary = save_policy_summary(
    agent,
    save_path='../results/policy_summary.json'
)

In [ ]:
# Test different learning rates
print("Testing hyperparameter sensitivity...\n")

import os

configs_to_test = [
    {'lr': 0.05, 'gamma': 0.99, 'label': 'lr=0.05'},
    {'lr': 0.10, 'gamma': 0.99, 'label': 'lr=0.10'},
    {'lr': 0.20, 'gamma': 0.99, 'label': 'lr=0.20'},
    {'lr': 0.10, 'gamma': 0.95, 'label': 'γ=0.95'},
    {'lr': 0.10, 'gamma': 0.90, 'label': 'γ=0.90'},
]

results_sensitivity = {}
n_test_episodes = 1000

for cfg in configs_to_test:
    test_config = {
        **QL_CONFIG,
        'learning_rate': cfg['lr'],
        'discount_factor': cfg['gamma'],
        'n_episodes': n_test_episodes
    }

    test_agent = QLearningAgent(
        DynamicPricingEnv(),
        test_config
    )

    test_rewards = test_agent.train(
        n_episodes=n_test_episodes,
        verbose=False
    )

    mean_rev = np.mean(test_rewards[-100:])
    results_sensitivity[cfg['label']] = mean_rev

    print(f"{cfg['label']:<12}: ${mean_rev:.0f}")

# --------------------------------------------------
# Create results directory automatically
# --------------------------------------------------

PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "hyperparam_sensitivity.png"
)

# --------------------------------------------------
# Plot
# --------------------------------------------------

plt.figure(figsize=(10, 5))

plt.bar(
    list(results_sensitivity.keys()),
    list(results_sensitivity.values()),
    color="steelblue",
    edgecolor="black"
)

plt.title(
    "Hyperparameter Sensitivity\nMean Revenue",
    fontweight="bold"
)

plt.ylabel("Mean Revenue ($)")
plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    bbox_inches="tight",
    dpi=150
)

print(f"✅ Saved: {SAVE_PATH}")

plt.show()

In [ ]:
best_config = max(
    results_sensitivity,
    key=results_sensitivity.get
)

print("╔══════════════════════════════════════════╗")
print("║    WEEK 1 DAY 4 — Q-TABLE ANALYSIS       ║")
print("╠══════════════════════════════════════════╣")
print("║  Q-TABLE INSIGHTS:                       ║")
print(f"║  Table Shape : 51×31×6 = 9,486 entries  ║")
print("║                                          ║")
print("║  LEARNED BEHAVIORS:                      ║")
print("║  ✅ Discounts near deadline              ║")
print("║  ✅ Premium pricing for low inventory    ║")
print("║  ✅ High prices early in season          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Best Hyperparam: {best_config:<23} ║")
print("╠══════════════════════════════════════════╣")
print("║  Q-Learning LIMITATION:                  ║")
print("║  ❌ Only works for SMALL state spaces    ║")
print("║  ❌ Real world = continuous states       ║")
print("║  → Solution: DQN (next week!) 🧠        ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Week 1 Wrap Up 📝            ║")
print("╚══════════════════════════════════════════╝")